# 학습 파이프라인 구축

Pytorch를 사용하여 학습 파이프라인을 구축하는 과제 노트북입니다.

진행 순서는 다음과 같습니다.

1. 데이터 다운로드 후 전처리
2. Pytorch Dataset 생성
3. Pytorch DataLoader 생성
4. 모델 구축
5. 모델 학습

## 1. 데이터를 다운로드 후 전처리
    - 예측 값 y는 성별(sex)와 생존여부(survived)이며 입력 데이터 X에 포함되어선 안 됩니다.
    - 입력할 데이터 X에 포함될 수치형 데이터는 평균이 0, 분산이 1인 표준편차가 되어야합니다.
    - 입력할 데이터 X에 범주형 데이터를 사용할 경우 class로 변환되어야합니다.
    - Train 데이터와 Test 데이터를 8:2로 분할하고 만일 Validation 데이터까지 포함한다면 8:1:1 비율로 나누십시오.

### 데이터 다운로드

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns

In [3]:
df = sns.load_dataset("titanic")
print(df.shape)
df.head()

(891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


### 정보 누설 방지 및 데이터 전처리

In [6]:
# [[YOUR CODE]]
# 수치형이 아닌 모든 항목의 컬럼 확인

cat_cols = df.select_dtypes(exclude=['int64', 'float64']).columns

for col in cat_cols:
    print(f"\n[{col}]")
    print(df[col].unique())
    df_ml = df.copy()

# 성인 컬럼 생성
df_ml['is_adult'] = (df_ml['who'] != 'child').astype(int)

# 정보 누설 및 중복 컬럼 제거
df_ml = df_ml.drop(
    columns=[
        'who',
        'adult_male',
        'alive',
        'class',
        'embark_town'
    ]
)

df_ml.head()
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 결측치 처리
df_ml['embarked'] = df_ml['embarked'].fillna('Unknown')
df_ml['deck'] = (
    df_ml['deck']
    .astype('object')
    .fillna('Unknown')
)

# sex를 타겟으로 사용하기 위해 먼저 수치화
df_ml['sex'] = df_ml['sex'].map({
    'female': 0,
    'male': 1
})

# bool → int
df_ml['alone'] = df_ml['alone'].astype(int)

# 범주형 Label Encoding
categorical_cols = [
    'embarked',
    'deck',
]

encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_ml[col] = le.fit_transform(df_ml[col])
    encoders[col] = le

# X, y 분리
X = df_ml.drop(columns=['sex', 'survived'])
y = df_ml[['sex', 'survived']]

# 수치형 변수 표준화
numeric_cols = [
    'pclass',
    'age',
    'sibsp',
    'parch',
    'fare'
]

scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

df_ml.head()


[sex]
['male' 'female']

[embarked]
['S' 'C' 'Q' nan]

[class]
['Third', 'First', 'Second']
Categories (3, object): ['First', 'Second', 'Third']

[who]
['man' 'woman' 'child']

[adult_male]
[ True False]

[deck]
[NaN, 'C', 'E', 'G', 'D', 'A', 'B', 'F']
Categories (7, object): ['A', 'B', 'C', 'D', 'E', 'F', 'G']

[embark_town]
['Southampton' 'Cherbourg' 'Queenstown' nan]

[alive]
['no' 'yes']

[alone]
[False  True]


,survived,pclass,sex,age,sibsp,parch,fare,embarked,deck,alone,is_adult
0,0,3,1,22.0,1,0,7.2500,2,7,0,1
1,1,1,0,38.0,1,0,71.2833,0,2,0,1
2,1,3,0,26.0,0,0,7.9250,2,7,1,1
3,1,1,0,35.0,1,0,53.1000,2,2,0,1
4,0,3,1,35.0,0,0,8.0500,2,7,1,1


### feature 선택

In [10]:
# [[YOUR CODE]]
# survived와 상관 관계

corr_survived = (
    df_ml.corr(numeric_only=True)['survived']
    .drop('survived')
    .drop('sex')
    .sort_values(key=abs, ascending=False)
)

print(corr_survived.head(15))
# sex와 상관 관계

corr_sex = (
    df_ml.corr(numeric_only=True)['sex']
    .drop('sex')
    .drop('survived')
    .sort_values(key=abs, ascending=False)
)

print(corr_sex.head(15))
selected_features = [
    'pclass',
    'deck',
    'fare',
    'alone',
    'embarked',
    'is_adult',
    'parch'
]

X = df_ml[selected_features]

y = df_ml[['survived', 'sex']]

pclass     -0.338481
deck       -0.294804
fare        0.257307
alone      -0.203367
embarked   -0.163517
is_adult   -0.136107
parch       0.081629
age        -0.077221
sibsp      -0.035322
Name: survived, dtype: float64
alone       0.303646
parch      -0.245489
fare       -0.182333
pclass      0.131900
deck        0.118282
sibsp      -0.114631
is_adult    0.111141
embarked    0.104057
age         0.093254
Name: sex, dtype: float64


### train / vaildation / test 데이터 분리

In [16]:
# [[YOUR CODE]]
from sklearn.model_selection import train_test_split

# train 80%
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# validation 10%, test 10%
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42
)

print(X_train.shape)
print(X_valid.shape)
print(X_test.shape)
print(y_train.shape)
print(y_valid.shape)
print(y_test.shape)

(712, 7)
(89, 7)
(90, 7)
(712, 2)
(89, 2)
(90, 2)


## 2. Pytorch Dataset 생성


In [12]:
# [[YOUR CODE]]
import torch
from torch.utils.data import Dataset, DataLoader

class TitanicDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(
            X.values,
            dtype=torch.float32
        )

        self.sex = torch.tensor(
            y["sex"].values,
            dtype=torch.float32
        ).unsqueeze(1)

        self.survived = torch.tensor(
            y["survived"].values,
            dtype=torch.float32
        ).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):

        return {
            "inputs": self.X[idx],
            "sex": self.sex[idx],
            "survived": self.survived[idx]
        }

train_ds = TitanicDataset(X_train, y_train)
val_ds   = TitanicDataset(X_valid, y_valid)
test_ds  = TitanicDataset(X_test, y_test)

# 출력 형식 확인
print(train_ds[0])

{'inputs': tensor([ 1.0000,  2.0000, 28.5000,  1.0000,  2.0000,  1.0000,  0.0000]), 'sex': tensor([1.]), 'survived': tensor([0.])}


## 3. Pytorch DataLoader 생성

In [13]:
# [[YOUR CODE]]
BATCH_SIZE = 32

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))
print("input:", batch["inputs"].shape)
print("label:", batch["sex"].shape, batch["survived"].shape)

input: torch.Size([32, 7])
label: torch.Size([32, 1]) torch.Size([32, 1])


## 4. 모델 구축

In [14]:
# [[YOUR CODE]]
import torch.nn as nn

class MultiTaskMLP(nn.Module):
    def __init__(self, in_features=7):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(in_features, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
        )
        self.sex_head = nn.Sequential(
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid(),
        )
        self.sur_head = nn.Sequential(
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        h = self.shared(x)
        return self.sex_head(h), self.sur_head(h)

model = MultiTaskMLP(in_features=X_train.shape[1])
model

MultiTaskMLP(
  (shared): Sequential(
    (0): Linear(in_features=7, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
  )
  (sex_head): Sequential(
    (0): Linear(in_features=16, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=1, bias=True)
    (3): Sigmoid()
  )
  (sur_head): Sequential(
    (0): Linear(in_features=16, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=1, bias=True)
    (3): Sigmoid()
  )
)

## 5. 모델 학습

### Loss function과 optimizer 설정

In [17]:
# [[YOUR CODE]]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
criterion = nn.BCELoss()  # Sigmoid 출력이라 BCELoss 사용
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 50

### 학습 루프 함수 작성

In [27]:
from tqdm.notebook import tqdm

def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    total_loss = 0.0
    correct_sex = 0
    correct_survived = 0
    total_count = 0

    pbar = tqdm(
        loader,
        desc="Train",
        leave=False
    )

    for batch in pbar:

        X = batch["inputs"].to(device)

        y_sex = batch["sex"].to(device)
        y_survived = batch["survived"].to(device)

        pred_sex, pred_survived = model(X)

        loss = (
            criterion(pred_sex, y_sex) +
            criterion(pred_survived, y_survived)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X.size(0)

        correct_sex += (
            (pred_sex > 0.5) == y_sex
        ).sum().item()

        correct_survived += (
            (pred_survived > 0.5) == y_survived
        ).sum().item()

        total_count += X.size(0)

        pbar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return {
        "loss": total_loss / total_count,
        "sex_acc": correct_sex / total_count,
        "survived_acc": correct_survived / total_count,
    }


### 모델 학습 실행

In [28]:
history = {
    "train_loss": [],
    "train_sex_acc": [],
    "train_survived_acc": [],
    "valid_loss": [],
    "valid_sex_acc": [],
    "valid_survived_acc": [],
}

for epoch in range(EPOCHS):

    train_metrics = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    valid_metrics = evaluate(
        model,
        val_loader,
        criterion,
        device
    )

    history["train_loss"].append(train_metrics["loss"])
    history["train_sex_acc"].append(train_metrics["sex_acc"])
    history["train_survived_acc"].append(train_metrics["survived_acc"])

    history["valid_loss"].append(valid_metrics["loss"])
    history["valid_sex_acc"].append(valid_metrics["sex_acc"])
    history["valid_survived_acc"].append(valid_metrics["survived_acc"])

    print(
        f"[{epoch+1:03d}/{EPOCHS}] "
        f"Train Loss: {train_metrics['loss']:.4f} | "
        f"Train Sex Acc: {train_metrics['sex_acc']:.4f} | "
        f"Train Survived Acc: {train_metrics['survived_acc']:.4f} | "
        f"Valid Loss: {valid_metrics['loss']:.4f} | "
        f"Valid Sex Acc: {valid_metrics['sex_acc']:.4f} | "
        f"Valid Survived Acc: {valid_metrics['survived_acc']:.4f}"
    )

Train:   0%|          | 0/23 [00:00<?, ?it/s]

Eval:   0%|          | 0/3 [00:00<?, ?it/s]

[001/50] Train Loss: 1.6586 | Train Sex Acc: 0.3525 | Train Survived Acc: 0.3820 | Valid Loss: 1.3441 | Valid Sex Acc: 0.3371 | Valid Survived Acc: 0.4607


Train:   0%|          | 0/23 [00:00<?, ?it/s]

Eval:   0%|          | 0/3 [00:00<?, ?it/s]

...

Eval:   0%|          | 0/3 [00:00<?, ?it/s]

[049/50] Train Loss: 1.1558 | Train Sex Acc: 0.7037 | Train Survived Acc: 0.6868 | Valid Loss: 1.1530 | Valid Sex Acc: 0.7079 | Valid Survived Acc: 0.6854


Train:   0%|          | 0/23 [00:00<?, ?it/s]

Eval:   0%|          | 0/3 [00:00<?, ?it/s]

[050/50] Train Loss: 1.1553 | Train Sex Acc: 0.7037 | Train Survived Acc: 0.6812 | Valid Loss: 1.1561 | Valid Sex Acc: 0.6966 | Valid Survived Acc: 0.6854
